# S6E10 — 14 RealMLP-rich + auxiliary features

> **Pipeline position:** 13 aux features → **14 RealMLP-rich + aux** → 11 stack

| | |
|---|---|
| **Input** | `X_rich_*.parquet` (09), `X_aux_*.parquet` (13), `y_train.parquet`; fold-0 checkpoint of the reference run (12) |
| **Output** | `preds/realmlp_rich_aux_folds[_s<seed>]/` (fold checkpoints); if accepted: `preds/oof_realmlp_rich_aux*.npy`, `test_realmlp_rich_aux*.npy` |
| **Runtime** | ≈ 2 min per fold on an RTX 3070 Ti (4 networks) |
| **Result** | see *Results & decisions* |

**Why:** `13` built 32 label-free columns (expected value of 16 columns from the other 20, and the deviation from it). The question is whether RealMLP ranks rows better with them. **The only change versus the reference run of `12`** is the 32 extra input columns: same recipe (the one of `10`; the v2 recipe was rejected in `12`), same 4 networks, same seed 0, same machine. So the benchmark is the paired reference **0.96138** (fold 0, GPU, seed 0), not the Mac numbers.

**Acceptance:** fold 0 first; a gain below +0.0002 is noise. If it passes: all 5 folds, the stack (`11`), and a transfer check with LightGBM.

> **Glossary:** *Paired benchmark* = the reference trained under identical conditions (data split, seed, machine), so the difference is attributable to the change. *Ablation* = removing one block of features to see what it contributed.

In [1]:
import torch          # first import: on the Mac the kernel crashes if torch comes after numpy/sklearn (OpenMP conflict)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np, pandas as pd, time, gc, warnings
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from pytabkit import RealMLP_TD_Classifier
warnings.filterwarnings('ignore')

SMOKE = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_ENS = 4            # as in the paired reference of 12
SEED = 0
AUX_SET = 'all'      # 'all' = 32 columns | 'ev' = expected values only | 'dev' = deviations only (ablation, if needed)
print('device:', DEVICE, '| networks:', N_ENS, '| seed:', SEED, '| aux set:', AUX_SET)

N_FOLDS, RANDOM_STATE = 5, 42
CANDS = [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working'), Path('.')]
PROC_DIR = next(p for p in CANDS if (p / 'X_aux_train.parquet').exists())
PREDS_DIR = next((p for p in [Path('preds'), Path('../preds'), Path('/kaggle/working/preds')] if p.exists()), Path('preds'))
X = pd.read_parquet(PROC_DIR / 'X_rich_train.parquet'); X_test = pd.read_parquet(PROC_DIR / 'X_rich_test.parquet')
A = pd.read_parquet(PROC_DIR / 'X_aux_train.parquet'); A_test = pd.read_parquet(PROC_DIR / 'X_aux_test.parquet')
AUX_COLS = [c for c in A.columns if AUX_SET == 'all' or c.startswith(f'aux_{AUX_SET}_')]
assert len(A) == len(X) and len(A_test) == len(X_test)
X = pd.concat([X, A[AUX_COLS]], axis=1); X_test = pd.concat([X_test, A_test[AUX_COLS]], axis=1)
y = pd.read_parquet(PROC_DIR / 'y_train.parquet')['y'].astype(int).values
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE).split(X, y))
assert list(X.columns) == list(X_test.columns)
TAG = 'rich_aux' if AUX_SET == 'all' else f'rich_aux_{AUX_SET}'      # checkpoint / file name
print(X.shape, X_test.shape, '| aux columns:', len(AUX_COLS), '| tag:', TAG)

C:\Users\89671\anaconda3\envs\kaggle-playground\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


W1010 03:48:51.537000 26272 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


device: cuda | networks: 4 | seed: 0 | aux set: all


(699635, 124) (299844, 124) | aux columns: 32 | tag: rich_aux


## 1. Input frame and recipe (identical to `10` / the reference of `12`)
The aux columns are plain numbers and go through RealMLP's own preprocessing (robust scaling, smooth clipping, PLR embeddings) like every other numeric column.

In [2]:
RATINGS = ['inflight_wifi_service','departure/arrival_time_convenient','ease_of_online_booking','gate_location','food_and_drink',
           'online_boarding','seat_comfort','inflight_entertainment','on-board_service','leg_room_service','baggage_handling',
           'checkin_service','cleanliness']
TWINS = {'route_id': 'flight_distance', 'age_cat': 'age', **{f'{r}_cat': r for r in RATINGS}}
FULL = pd.concat([X, X_test], ignore_index=True)
TWIN_LEVELS = {t: sorted(FULL[s].astype(int).unique()) for t, s in TWINS.items()}
KEYS = [c for c in X.columns if c.startswith('key_')]
TE_SOURCES = [c for c in ['flight_distance', 'age', 'fd_div10', 'fd_mod1000'] + KEYS if c in X.columns]
CAT_COLS = ['gender', 'customer_type', 'type_of_travel', 'class'] + list(TWINS)

def nn_frame(D):
    D = D.drop(columns=['orig_prob'] + KEYS).copy()
    for t, s in TWINS.items():
        D[t] = pd.Categorical(D[s].astype(int), categories=TWIN_LEVELS[t])
    assert not D.isna().any().any()
    return D

PARAMS = dict(
    n_epochs=1 if SMOKE else 4, n_ens=2 if SMOKE else N_ENS, batch_size=256, use_early_stopping=False,
    val_metric_name='1-auc_ovr', device=DEVICE, random_state=0, verbosity=0,
    lr=0.053, wd=0.0236, sq_mom=0.988, lr_sched='lin_cos_log_15', wd_sched='cos_log_15',
    first_layer_lr_factor=0.25, embedding_size=5, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act='silu', p_drop=0.05, p_drop_sched='expm4t',
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name='gelu', plr_lr_factor=0.1151, plr_sigma=2.33,
    use_ls=True, ls_eps=0.01, ls_eps_sched='sqrt_cos', add_front_scale=False,
    bias_init_mode='neg-uniform-dynamic-2',
    tfms=['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'],
)

# In-fold target encoding: fitted on the fold's TRAINING part only (inner cv=5), applied to validation/test.
def add_te(Xtr, ytr, others):
    te = TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=42)
    cols = [f'te_{c}' for c in TE_SOURCES]
    tr_te = pd.DataFrame(te.fit_transform(Xtr[TE_SOURCES].astype(int), ytr), columns=cols, index=Xtr.index).astype('float32')
    outs = [pd.DataFrame(te.transform(o[TE_SOURCES].astype(int)), columns=cols, index=o.index).astype('float32') for o in others]
    return [pd.concat([D, T], axis=1) for D, T in zip([Xtr] + list(others), [tr_te] + outs)]

def fit_predict(tr_idx, va_idx, seed=None, rows=None):
    seed = SEED if seed is None else seed
    tr_idx = tr_idx if rows is None else tr_idx[:rows]
    Xtr, Xva, Xte = add_te(X.iloc[tr_idx], y[tr_idx], [X.iloc[va_idx], X_test])
    Xtr, Xva, Xte = nn_frame(Xtr), nn_frame(Xva), nn_frame(Xte)
    model = RealMLP_TD_Classifier(**{**PARAMS, 'random_state': seed})
    model.fit(Xtr, y[tr_idx], X_val=Xva, y_val=y[va_idx], cat_col_names=CAT_COLS)
    p_va, p_te = model.predict_proba(Xva)[:, 1], model.predict_proba(Xte)[:, 1]
    del model; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return p_va, p_te

def run_fold(k, seed=None):
    seed = SEED if seed is None else seed
    d = PREDS_DIR / (f'realmlp_{TAG}_folds' if seed == 0 else f'realmlp_{TAG}_folds_s{seed}')
    fo, ft = d / f'oof_{k}.npy', d / f'test_{k}.npy'
    if fo.exists() and ft.exists(): return np.load(fo), np.load(ft), None
    t0 = time.time(); p_va, p_te = fit_predict(*FOLDS[k], seed=seed, rows=60000 if SMOKE else None)
    if not SMOKE: d.mkdir(exist_ok=True); np.save(fo, p_va); np.save(ft, p_te)
    return p_va, p_te, (time.time() - t0) / 60

## 2. Fold 0 — paired benchmark
Reference: RealMLP-rich without aux, same recipe, seed 0, this machine — the checkpoint `preds/realmlp_rich_refgpu_folds/oof_0.npy` from `12` (0.96138).

In [3]:
va0 = FOLDS[0][1]
p_va0, p_te0, mins = run_fold(0)
a_new = roc_auc_score(y[va0], p_va0)
ref_f = PREDS_DIR / 'realmlp_rich_refgpu_folds' / 'oof_0.npy'
a_ref = roc_auc_score(y[va0], np.load(ref_f)) if ref_f.exists() else np.nan
print(f'fold 0: RealMLP-rich + aux ({AUX_SET}) {a_new:.5f}' + ('' if mins is None else f' ({mins:.1f} min)'))
print(f'paired reference (no aux)      {a_ref:.5f}\ndifference {a_new - a_ref:+.5f}   (acceptance: >= +0.0002)')

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


You are using a CUDA device ('NVIDIA GeForce RTX 3070 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


fold 0: RealMLP-rich + aux (all) 0.96138 (1.6 min)
paired reference (no aux)      0.96138
difference +0.00000   (acceptance: >= +0.0002)


## 3. All 5 folds (only if fold 0 passed)
Resumable from `preds/realmlp_<tag>_folds[_s<seed>]/`. Off by default.

In [4]:
RUN_ALL = False
if RUN_ALL and not SMOKE:
    oof = np.full(len(X), np.nan); tests = []
    for k, (tr, va) in enumerate(FOLDS):
        p_va, p_te, mins = run_fold(k)
        oof[va] = p_va; tests.append(p_te)
        print(f'fold {k}: AUC {roc_auc_score(y[va], p_va):.5f}' + ('' if mins is None else f'  ({mins:.1f} min)'), flush=True)
    print(f'\nOOF AUC (RealMLP {TAG}): {roc_auc_score(y, oof):.5f}')
    sfx = '' if SEED == 0 else f'_s{SEED}'
    np.save(PREDS_DIR / f'oof_realmlp_{TAG}{sfx}.npy', oof); np.save(PREDS_DIR / f'test_realmlp_{TAG}{sfx}.npy', np.mean(tests, axis=0))

## Results & decisions

| Run (fold 0, GPU, seed 0, 4 networks) | AUC | Difference to reference |
|---|---|---|
| reference: RealMLP-rich (from `12`) | 0.9613755 | – |
| **+ 32 aux columns (`13`)** | **0.9613804** | **+0.000005** |
| *(for comparison: recipe v2 from `12`)* | *0.9613780* | *+0.000002* |

**Decision: rejected, step closed.** The predictions do change (Spearman 0.994 with the reference, max |Δp| 0.29), but the ranking quality does not. An ablation (`AUX_SET='ev'` / `'dev'`) is not run: with a total effect of zero there is nothing to attribute.

**Interpretation:** the same lesson as the rating summaries in `02` — information that is a function of the 21 raw columns of the *same* row is already available to the network; only information about *other rows* (frequencies, target encodings, the original survey) moved the score in this project. On fold 0, RealMLP-rich sits on a plateau at ≈ 0.96138 for every variant tried here.

Not saved: OOF/test for all folds (only the fold-0 checkpoint in `preds/realmlp_rich_aux_folds/`).


## Glossary
* **Aux features** — expected value of a column given the other columns (`aux_ev_*`) and the deviation of the actual value from it (`aux_dev_*`); built without labels in `13`.
* **Paired benchmark** — reference run under identical conditions (split, seed, machine, recipe).
* **Ablation** — training without one block of features to measure its contribution.